# Daily visitor arrivals in Macau and Hong Kong: data pipeline

The task includes 27 Macau source-market series and two Hong Kong visitor-category series. At each origin, only the available history is used to forecast the next 28 days. Source data are read-only, and zero observations are retained. Set the working directory to the repository root before running this notebook. This notebook demonstrates data preparation and boundary checks; it does not reproduce every experiment reported from the server.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value.to_string() if hasattr(value, 'to_string') else value)
PROJECT = Path.cwd()
CODE = PROJECT / 'track1' / 'code'
assert CODE.is_dir(), 'Set the working directory to the repository root.'
sys.path.insert(0, str(CODE))
from data_pipeline import (
    MACAU_DIR, HONG_KONG_WORKBOOK, TEST_START, TEST_END,
    load_all, quality_summary, make_input_task,
    write_data_snapshot, write_task_artifacts,
)
from protocol_config import load_protocol
print('Macau:', MACAU_DIR)
print('Hong Kong workbook:', HONG_KONG_WORKBOOK)
print('Target-date test period:', TEST_START, 'to', TEST_END)


## 1. Load and validate the data

Macau data are read from the `data,Tourism` CSV sources. Hong Kong data are read from the workbook's daily-data worksheet: mainland visitors and other visitors are separate target series. The workbook's total visitor count is used only for reconciliation. The loader retains the original source worksheet and column identifiers.

In [ ]:
series = load_all()
qa = pd.DataFrame(quality_summary(series))
display(qa.groupby('destination').agg(series=('market','count'),
                                      first_date=('first_date','min'),
                                      last_date=('last_date','max'),
                                      total_days=('days','sum'),
                                      zero_days=('zero_days','sum')))
display(qa.head(4))
assert len(series) == 29
assert (qa['days'] > 0).all()


## 2. Generate normalized snapshots, model inputs and isolated targets

`task_inputs.parquet` contains no target observations; `targets.parquet` is reserved for the independent evaluator. The manifest retains logical source identifiers and SHA256 hashes rather than machine-specific absolute paths.

In [ ]:
manifest = write_data_snapshot(PROJECT / 'shared' / 'data')
task_manifest = write_task_artifacts(series, PROJECT / 'shared' / 'data')
print('series:', manifest['series_count'])
print('normalized observations:', sum(item['days'] for item in manifest['quality']))
print('source files hashed:', len(manifest['source_sha256']))
print('isolated tasks:', task_manifest['task_count'])
print('split task counts:', task_manifest['split_task_counts'])


## 3. Inspect the three configured sets of forecast origins

`shared/configs/protocol.yaml` explicitly lists A01–A13 (pandemic stress), B01–B12 (tuning) and O01–O33 (outer evaluation). Sets A and B include Macau only; Hong Kong participates in outer evaluation only. Listing these origins does not by itself establish when a configuration was frozen.

In [ ]:
protocol = load_protocol()
for split in ('pandemic_stress', 'recovery_tuning', 'outer_backtest'):
    folds = [fold for fold in protocol.folds if fold.split_name == split]
    print(split, len(folds), folds[0].fold_id, folds[0].origin,
          'to', folds[-1].fold_id, folds[-1].origin, '| destinations:', folds[0].destinations)


## 4. Check the model input boundary

In this O33 example, `task.history` ends on 2026-06-14, and the task object has no `actual` attribute. Target observations are stored separately in `targets.parquet`.

In [ ]:
from datetime import date
item = series['Macau:CN_Chinese_mainland']
task = make_input_task(item, date(2026, 6, 14), history_length=180)
print(task.key, '| history:', task.history_start, 'to', task.origin, len(task.history),
      '| forecast dates:', task.target_dates[0], 'to', task.target_dates[-1])
assert len(task.history) == 180 and len(task.target_dates) == 28
assert not hasattr(task, 'actual')
assert task.target_dates[0] > task.origin
assert task.target_dates[-1] <= TEST_END
assert np.array_equal(task.history, item.values.loc[str(task.history_start):str(task.origin)].to_numpy())


Next, run `02_models_and_backtesting.ipynb`. This notebook checks data structure and temporal boundaries; it does not train models.